# TD: RAG

Dans ce notebook, un RAG basique est implémenté:
- On chunk les documents par paragraphes
- On a un embedding pour les chunks
- Pour une question, on peut embedde la question et récupérer les N chunks les plus pertinents
- On utilise un modèle de génération de texte (SMoLL) pour faire la partie question + chunks les plus pertinents -> réponse.

Téléchargez (cette archive)[https://drive.google.com/file/d/1TnfKs7bTwmpbXklbgiIBpdw7I_wJ5y9Y/view?usp=sharing] avec différentes 

Dans ce TD, vous allez expérimenter différentes façons de chunk et d'embeded les documents et les questions pour que le RAG retrieve les documents les plus pertinents. <br/>
Vous expérimenterez aussi la prompt donnée au générateur de texte pour avoir les meilleures réponses.

A rendre:
- un CSV avec les documents chunkés et leur embedding. Le CSV a 2 colonnes "chunk", "embedding". L'embedding, dans ce CSV, d'un document D doit être le JSON de la liste de float. Autrement dit, quand je lis votre CSV avec pandas, la ligne json.loads(df.loc[0, "embedding"]) doit retourner une liste de floats
- un CSV avec les questions embedded. Je me servirais des 2 CSVs pour calculer la mean reciprocal rank de la partie retrieval de votre RAG.
- un CSV avec query,reply contenant, pour chaque query, la réponse du RAG
- Le notebook avec toutes les étapes de votre RAG (chunk, embedding, prompt)

In [8]:
import numpy as np

import pandas as pd
from pathlib import Path

# Data loading

In [2]:
path = Path("../data/raw/rag/")

In [3]:
texts = []
for filename in path.glob("*.md"):
    with open(filename) as f:
        texts.append(f.read())

texts[0]

'# Title: Introduction to Internet of Things (IoT)  \n\n**Teacher:** Dr. Rebecca Tan  \n\n**Description:**  \nThis course explores the fundamentals of the Internet of Things (IoT), focusing on the design, development, and deployment of connected devices. Students will learn about IoT architectures, sensors, communication protocols, data processing, and security challenges. Through hands-on projects, students will gain experience building IoT applications using microcontrollers, cloud platforms, and IoT-specific technologies.  \n\n**Prerequisites:**  \n- Basic programming skills (Python or C preferred)  \n- Understanding of networking basics  \n- Completion of "Introduction to Computer Science" or equivalent  \n\n**Assessment:**  \n- Weekly lab assignments (30%)  \n- Midterm exam: IoT principles and architectures (20%)  \n- Final project: Develop and present a functional IoT prototype (40%)  \n- Class participation and discussions (10%)  \n\n**Schedule Time:**  \n- Tuesdays and Thursday

# Chunk
## Basic

In [4]:
def parse_class(text):
    chunks = text.split("\n\n")
    title = chunks[0].replace("# Title: ", "")
    return {"title": title, "chunks": chunks}

In [5]:
def parse_class_add_title(text):
    chunks = text.split("\n\n")
    title = chunks[0].replace("# Title: ", "")
    return {"title": title, "chunks": [f"{title}: {chunk}" for chunk in chunks]}

In [6]:
chunks = sum((parse_class_add_title(txt)["chunks"] for txt in texts), [])

# Embedding

## BAAI's embedding

In [9]:
from FlagEmbedding import FlagModel

2025-01-06 10:31:28.096792: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-06 10:31:28.097635: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-06 10:31:28.098671: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-06 10:31:28.256502: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [10]:
model = FlagModel(
    'BAAI/bge-base-en-v1.5',
    query_instruction_for_retrieval="Represent this sentence for searching relevant passages:",
    use_fp16=True,
)

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [11]:
corpus_embedding = model.encode(chunks)

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [12]:
queries = [
    "Who is the reinforcement learning teacher?",
    "In what class will I learn game AI?",
]

In [13]:
query_embedding = model.encode(queries)

In [14]:
sim_scores = query_embedding @ corpus_embedding.T

In [15]:
for query, score in zip(queries, sim_scores):
    print(" ---- ")
    print("Query: ", query)
    indexes = np.argsort(score)[-5:]
    print("Sources:")
    for i, idx in enumerate(reversed(indexes)):
        if score[idx] > .5:
            print(f"{i+1} -- similarity {score[idx]:.2f} -- \"", chunks[idx], '"')
            
    

 ---- 
Query:  Who is the reinforcement learning teacher?
Sources:
1 -- similarity 0.80 -- " Foundations of Reinforcement Learning  : **Teacher:** Dr. Arjun Patel   "
2 -- similarity 0.74 -- " Foundations of Reinforcement Learning  : # Title: Foundations of Reinforcement Learning   "
3 -- similarity 0.71 -- " Foundations of Reinforcement Learning  : 2. **Tabular Methods**  
   - Dynamic programming approaches: Policy Iteration and Value Iteration  
   - Monte Carlo methods and Temporal-Difference (TD) Learning   "
4 -- similarity 0.71 -- " Foundations of Reinforcement Learning  : 4. **Policy-Based Methods**  
   - Policy Gradient methods and REINFORCE algorithm  
   - Advantage Actor-Critic (A2C) and Proximal Policy Optimization (PPO)   "
5 -- similarity 0.71 -- " Foundations of Reinforcement Learning  : **Description:**  
This course explores the foundational principles and practical applications of reinforcement learning (RL), a branch of machine learning focused on decision-making a

# Eval retrieval: Mean Reciprocal Rank
Le fichier [question_answer_short.csv](https://drive.google.com/file/d/1EB8IwGlqvpNy3oq7xyR2IzdqJDX8C_fr/view?usp=drive_link) contient une liste de question et le texte à retrouver dans les documents.<br/>
Je considère que tout chunk contenant le "texte à retrouver" était un bon chunk

In [17]:
df = pd.read_csv(path / "question_answer_short.csv")

In [18]:
query_embedding = model.encode(list(df["question"]))

In [19]:
acceptable_chunks = []
for answer in df["answer"]:
    chunks_ok = set(i for i, chunk in enumerate(chunks) if answer in chunk)
    acceptable_chunks.append(chunks_ok)

In [20]:
def compute_mrr(sim_score, acceptable_chunks):
    ranks = []
    for this_score, this_acceptable_chunks in zip(sim_score, acceptable_chunks):
        indexes = reversed(np.argsort(this_score))
        rank = 1 + next(i for i, idx in enumerate(indexes) if idx in this_acceptable_chunks)
        ranks.append(rank)
        
    return {
        "score": sum(1 / r if r < 6 else 0 for r in ranks) / len(ranks),
        "ranks": ranks,
    }

In [21]:
sim_scores = query_embedding @ corpus_embedding.T

In [22]:
res = compute_mrr(sim_scores, acceptable_chunks)
res["score"]

0.6

# Text generation

In [23]:
def get_context(query, corpus, corpus_embeddings):
    query_embedding = model.encode([query])
    sim_scores = query_embedding @ corpus_embedding.T
    indexes = list(np.argsort(sim_scores[0]))[-5:]
    return [corpus[i] for i in indexes]

In [24]:
get_context("Which class will teach me to build a chatbot?", chunks, corpus_embedding)

['# Natural Language Processing (NLP) Fundamentals and Applications: 5. **Applications of NLP**\n  - Sentiment analysis and text classification\n  - Machine translation and summarization\n  - Chatbots and conversational agents',
 '# Natural Language Processing (NLP) Fundamentals and Applications: **Description:**\nThis course offers a comprehensive introduction to the field of Natural Language Processing (NLP), focusing on the computational techniques that allow machines to understand, interpret, and generate human language. You will learn about linguistic structures, text preprocessing, sentiment analysis, machine translation, and language modeling. Using hands-on projects and industry-relevant tools, this course provides a strong foundation in both traditional and modern NLP methods, including neural networks and transformers.',
 '# Natural Language Processing (NLP) Fundamentals and Applications: Whether you aim to pursue a career in AI or enhance your programming toolkit, this cours

## SMOLL

In [25]:
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint = "HuggingFaceTB/SmolLM2-360M-Instruct"
# checkpoint = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

device = "cpu" # for GPU usage or "cpu" for CPU usage

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model_generator = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [26]:
def build_smoll_prompt(query, corpus, corpus_embedding):
    context_str = "\n\n".join(get_context(query, chunks, corpus_embedding))

    prompt = f"""<|im_start|>system
You reply to the user's request using only context information.
Context information to answer "{query}" is below
------
Context:
{context_str}
------
You are a helpful assistant for a Computer Science university. You reply to students'questions about the courses that they can attend.
<|im_end|>
<|im_start|>user
{query}
<|im_reend|>
"""
    return prompt


In [27]:
def build_smoll_messages(query, chunks, corpus_embedding):
    context_str = "\n\n".join(get_context(query, chunks, corpus_embedding))

    messages = [
        {"role": "system", "content": f"""You reply to the user's request using only context information.
Context information to answer "{query}" is below
------
Context:
{context_str}
------
You are a helpful assistant for a Computer Science university. You reply to students'questions about the courses that they can attend.
"""},
        {"role": "user", "content": query},
    ]

    return messages


In [28]:
messages = build_smoll_messages("Who is the NLP teacher?", chunks, corpus_embedding)

input_text=tokenizer.apply_chat_template(messages, tokenize=False)
inputs = tokenizer.encode(input_text, return_tensors="pt").to(device)
outputs = model_generator.generate(inputs, max_new_tokens=100, temperature=0.01, top_p=0.9, do_sample=True)
print(tokenizer.decode(outputs[0]))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|im_start|>system
You reply to the user's request using only context information.
Context information to answer "Who is the NLP teacher?" is below
------
Context:
# Natural Language Processing (NLP) Fundamentals and Applications: **Course Outline:**
1. **Introduction to NLP**
  - Key concepts and challenges
  - Overview of linguistic structure and grammar

# Natural Language Processing (NLP) Fundamentals and Applications: **Prerequisites:**
- Proficiency in Python programming
- Basic understanding of linear algebra and probability
- Successful completion of "Introduction to Machine Learning" or equivalent

# Natural Language Processing (NLP) Fundamentals and Applications: # Natural Language Processing (NLP) Fundamentals and Applications

# Natural Language Processing (NLP) Fundamentals and Applications: **Schedule Time:**
- Tuesdays and Thursdays: 10:00 AM - 11:30 AM
- Lab Sessions: Fridays 2:00 PM - 4:00 PM

# Natural Language Processing (NLP) Fundamentals and Applications: **Teacher

# OpenAI generator
Si vous voulez utiliser OpenAI

In [43]:
from dotenv import load_dotenv
import os

load_dotenv()
openai_key = os.getenv("OPENAI_API_KEY")
mistral_key = os.getenv("MISTRAL_API_KEY")

In [41]:
import openai
import mistralai

In [44]:
client = mistralai.Mistral(api_key=mistral_key)

In [53]:
query = "What are the applications of NLP?"

context_str = "\n\n".join(get_context(query, chunks, corpus_embedding))

prompt = f"""Context information is below.
---------------------
{context_str}
---------------------
Given the context information and not prior knowledge, answer the query.
If the answer is not in the context information, reply "I cannot answer that question".
Query: {query}
Answer:"""

In [54]:
def run_mistral(user_message, model="mistral-large-latest"):
    messages = [
        {
            "role": "user", "content": user_message
        }
    ]
    chat_response = client.chat.complete(
        model=model,
        messages=messages
    )
    return chat_response.choices[0].message.content

print(run_mistral(prompt))

The applications of NLP mentioned in the context are:

- Sentiment analysis and text classification
- Machine translation and summarization
- Chatbots and conversational agents
